In [2]:
# ========================================
# PRICE PREDICTION - WIDE & DEEP TWO-ENCODER MODEL (5-FOLD CV with Scaling)
# ========================================
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
import gc
import os
from datetime import datetime

# reproducibility
torch.manual_seed(42)
np.random.seed(42)

# ======================
# 1. DEVICE
# ======================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ======================
# 2. LOAD ALL DATA
# ======================
print("Loading all image and text embeddings...")

all_img_emb = np.load("../embeddings/siglip2_train_image_embeddings.npy")
all_text_emb = np.load("../embeddings/all_text_emb.npy")

image_dim = all_img_emb.shape[1]
text_dim = all_text_emb.shape[1]
print(f"Detected Image embedding dimension: {image_dim}")
print(f"Detected Text embedding dimension: {text_dim}")

X_all = np.concatenate([all_img_emb, all_text_emb], axis=1)

all_df = pd.read_csv("../clean_data/train.csv")
y_all = all_df["price"].values.astype(np.float32)

# cleanup
del all_img_emb, all_text_emb, all_df
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

print(f"Final combined feature matrix shape: {X_all.shape}")
print(f"Final combined target array shape: {y_all.shape}")

# Optional: log-transform for long-tail prices
use_log = True
if use_log:
    print("Applying log1p transformation to target prices.")
    y_all = np.log1p(y_all).astype(np.float32)

# ======================
# 3. DATASET CLASS
# ======================
class PriceDataset(Dataset):
    def __init__(self, X, y):
        # Expect X as numpy float32 already
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y).float()  # shape (N,)
        
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ======================
# 4. MODEL ARCHITECTURE
# ======================
class WideDeepTwoEncoderMLP(nn.Module):
    def __init__(self, image_dim, text_dim, dropout_rate=0.3):
        super().__init__()
        self.image_dim = image_dim
        
        # Image encoder branch
        self.image_encoder = nn.Sequential(
            nn.Linear(image_dim, 1024), nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(512, 256), nn.ReLU()
        )
        # Text encoder branch
        self.text_encoder = nn.Sequential(
            nn.Linear(text_dim, 1024), nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(512, 256), nn.ReLU()
        )
        # Attention mechanisms (scalar attention per branch)
        self.image_attn = nn.Sequential(nn.Linear(256, 128), nn.Tanh(), nn.Linear(128, 1), nn.Sigmoid())
        self.text_attn = nn.Sequential(nn.Linear(256, 128), nn.Tanh(), nn.Linear(128, 1), nn.Sigmoid())
        # Wide path
        self.wide = nn.Linear(image_dim + text_dim, 64)
        # Final head - no activation so outputs can be any real number (suitable for log targets)
        self.out = nn.Linear(256 + 256 + 64, 1)

    def forward(self, x):
        # x shape: (B, image_dim + text_dim)
        wide_feat = self.wide(x)
        image_embs, text_embs = x[:, :self.image_dim], x[:, self.image_dim:]
        image_feat = self.image_encoder(image_embs)
        # attention returns (B,1) -> broadcast multiply
        image_att = self.image_attn(image_feat)
        image_feat = image_feat * image_att
        text_feat = self.text_encoder(text_embs)
        text_att = self.text_attn(text_feat)
        text_feat = text_feat * text_att
        deep_feat = torch.cat([image_feat, text_feat], dim=1)
        combined = torch.cat([deep_feat, wide_feat], dim=1)
        out = self.out(combined).squeeze(1)  # shape: (B,)
        return out

# ======================
# 5. SMAPE METRIC
# ======================
def smape(y_true, y_pred, eps=1e-8):
    # y_true and y_pred are tensors (same device)
    if use_log:
        # convert back to original scale for SMAPE
        y_true = torch.expm1(y_true)
        y_pred = torch.expm1(y_pred)
    numerator = torch.abs(y_pred - y_true)
    denominator = (torch.abs(y_true) + torch.abs(y_pred)) / 2 + eps
    return 100.0 * torch.mean(numerator / denominator)

# ======================
# 6. CROSS-VALIDATION & TRAINING
# ======================
N_SPLITS = 6
EPOCHS = 85
BATCH_SIZE = 64
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-5

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(kf.split(X_all), start=1):
    print("-" * 60)
    print(f"🚀 STARTING FOLD {fold}/{N_SPLITS}  - {datetime.now().isoformat()}")
    print("-" * 60)
    
    X_train, X_val = X_all[train_idx], X_all[val_idx]
    y_train, y_val = y_all[train_idx], y_all[val_idx]
    
    train_ds = PriceDataset(X_train, y_train)
    val_ds = PriceDataset(X_val, y_val)
    
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    
    model = WideDeepTwoEncoderMLP(image_dim=image_dim, text_dim=text_dim).to(device)
    curr_best_model = WideDeepTwoEncoderMLP(image_dim=image_dim, text_dim=text_dim).to(device)
    curr_best_model.load_state_dict(torch.load(f"./best 6fold/best_price_model_fold_{fold}.pt", map_location=device))
    print(f"Loaded current best model for fold {fold} for weighted training.")

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    loss_fn = nn.L1Loss()
    
    best_val_smape = float("inf")
    os.makedirs("weighted", exist_ok=True)
    model_save_path = f"weighted/best_price_model_fold_{fold}.pt"
    
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_train_loss = 0.0
        for X_batch, y_batch in train_dl:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            optimizer.zero_grad()
            preds = model(X_batch)
            loss = loss_fn(preds, y_batch) + loss_fn(preds, curr_best_model(X_batch)) * 0.3
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item() * X_batch.size(0)
        
        train_mae = total_train_loss / len(train_ds)
        
        # Validation
        model.eval()
        all_preds = []
        all_targets = []
        with torch.no_grad():
            for X_batch, y_batch in val_dl:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                preds = model(X_batch)
                all_preds.append(preds)
                all_targets.append(y_batch)
        all_preds_tensor = torch.cat(all_preds)
        all_targets_tensor = torch.cat(all_targets)
        
        val_smape = smape(all_targets_tensor, all_preds_tensor).item()
        
        # scheduler step (after epoch / validation)
        scheduler.step()
        
        print(f"Fold {fold}, Epoch {epoch:02d}: Train MAE={train_mae:.6f}, Val SMAPE={val_smape:.6f}")
        
        # Save best model
        if val_smape < best_val_smape:
            best_val_smape = val_smape
            torch.save(model.state_dict(), model_save_path)
            print(f"✅ Saved best model for fold {fold} to {model_save_path} (SMAPE: {best_val_smape:.6f})")
    
    # cleanup after fold
    del model, optimizer, scheduler, train_dl, val_dl, train_ds, val_ds
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

print("\n🎉 Cross-validation training complete.")

Using device: cuda
Loading all image and text embeddings...
Detected Image embedding dimension: 1536
Detected Text embedding dimension: 4096
Final combined feature matrix shape: (74999, 5632)
Final combined target array shape: (74999,)
Applying log1p transformation to target prices.
------------------------------------------------------------
🚀 STARTING FOLD 1/6  - 2025-10-13T23:03:50.644600
------------------------------------------------------------
Loaded current best model for fold 1 for weighted training.
Fold 1, Epoch 01: Train MAE=0.711427, Val SMAPE=52.295296
✅ Saved best model for fold 1 to weighted/best_price_model_fold_1.pt (SMAPE: 52.295296)
Fold 1, Epoch 02: Train MAE=0.633510, Val SMAPE=50.427181
✅ Saved best model for fold 1 to weighted/best_price_model_fold_1.pt (SMAPE: 50.427181)
Fold 1, Epoch 03: Train MAE=0.602129, Val SMAPE=48.903988
✅ Saved best model for fold 1 to weighted/best_price_model_fold_1.pt (SMAPE: 48.903988)
Fold 1, Epoch 04: Train MAE=0.578818, Val SMAP

In [3]:
# ========================================
# INFERENCE - WIDE & DEEP TWO-ENCODER MODEL (NO SCALING)
# ========================================
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
import gc
import os

class WideDeepTwoEncoderMLP(nn.Module):
    def __init__(self, image_dim, text_dim, dropout_rate=0.3):
        super().__init__()
        self.image_dim = image_dim
        
        # Image encoder branch
        self.image_encoder = nn.Sequential(
            nn.Linear(image_dim, 1024), nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(512, 256), nn.ReLU()
        )
        # Text encoder branch
        self.text_encoder = nn.Sequential(
            nn.Linear(text_dim, 1024), nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(512, 256), nn.ReLU()
        )
        # Attention mechanisms (scalar attention per branch)
        self.image_attn = nn.Sequential(nn.Linear(256, 128), nn.Tanh(), nn.Linear(128, 1), nn.Sigmoid())
        self.text_attn = nn.Sequential(nn.Linear(256, 128), nn.Tanh(), nn.Linear(128, 1), nn.Sigmoid())
        # Wide path
        self.wide = nn.Linear(image_dim + text_dim, 64)
        # Final head - no activation so outputs can be any real number (suitable for log targets)
        self.out = nn.Linear(256 + 256 + 64, 1)

    def forward(self, x):
        # x shape: (B, image_dim + text_dim)
        wide_feat = self.wide(x)
        image_embs, text_embs = x[:, :self.image_dim], x[:, self.image_dim:]
        image_feat = self.image_encoder(image_embs)
        # attention returns (B,1) -> broadcast multiply
        image_att = self.image_attn(image_feat)
        image_feat = image_feat * image_att
        text_feat = self.text_encoder(text_embs)
        text_att = self.text_attn(text_feat)
        text_feat = text_feat * text_att
        deep_feat = torch.cat([image_feat, text_feat], dim=1)
        combined = torch.cat([deep_feat, wide_feat], dim=1)
        out = self.out(combined).squeeze(1)  # shape: (B,)
        return out

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ------------------------------
# 1. LOAD TEST EMBEDDINGS
# ------------------------------
print("Loading test image and text embeddings...")

test_img_emb = np.load("../embeddings/updated_siglip2_test_image_embeddings_no_zero.npy")
test_text_emb = np.load("../embeddings/mistral_test_embeddings_from_notebook.npy")

image_dim = test_img_emb.shape[1]
text_dim = test_text_emb.shape[1]
print(f"Image embedding dim: {image_dim}")
print(f"Text embedding dim: {text_dim}")

X_test = np.concatenate([test_img_emb, test_text_emb], axis=1).astype(np.float32)
test_df = pd.read_csv("../clean_data/test.csv")

# Cleanup
del test_img_emb, test_text_emb
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

print(f"Final test matrix shape: {X_test.shape}")

# ------------------------------
# 2. DATASET CLASS
# ------------------------------
class PriceDataset(Dataset):
    def __init__(self, X):
        self.X = torch.from_numpy(X)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx]

BATCH_SIZE = 128

test_ds = PriceDataset(X_test)
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

Using device: cuda
Loading test image and text embeddings...
Image embedding dim: 1536
Text embedding dim: 4096
Final test matrix shape: (75000, 5632)


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model0 = WideDeepTwoEncoderMLP(image_dim=image_dim, text_dim=text_dim).to(device)
model0.load_state_dict(torch.load("./weighted/best_price_model_fold_1.pt", map_location=device))
model0.eval()
model1 = WideDeepTwoEncoderMLP(image_dim=image_dim, text_dim=text_dim).to(device)
model1.load_state_dict(torch.load("./weighted/best_price_model_fold_2.pt", map_location=device))
model1.eval()
model2 = WideDeepTwoEncoderMLP(image_dim=image_dim, text_dim=text_dim).to(device)
model2.load_state_dict(torch.load("./weighted/best_price_model_fold_3.pt", map_location=device))
model2.eval()
model3 = WideDeepTwoEncoderMLP(image_dim=image_dim, text_dim=text_dim).to(device)
model3.load_state_dict(torch.load("./weighted/best_price_model_fold_4.pt", map_location=device))
model3.eval()
model4 = WideDeepTwoEncoderMLP(image_dim=image_dim, text_dim=text_dim).to(device)
model4.load_state_dict(torch.load("./weighted/best_price_model_fold_5.pt", map_location=device))
model4.eval()
model5 = WideDeepTwoEncoderMLP(image_dim=image_dim, text_dim=text_dim).to(device)
model5.load_state_dict(torch.load("./weighted/best_price_model_fold_6.pt", map_location=device))
model5.eval()

WideDeepTwoEncoderMLP(
  (image_encoder): Sequential(
    (0): Linear(in_features=1536, out_features=1024, bias=True)
    (1): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=1024, out_features=512, bias=True)
    (5): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=512, out_features=256, bias=True)
    (9): ReLU()
  )
  (text_encoder): Sequential(
    (0): Linear(in_features=4096, out_features=1024, bias=True)
    (1): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=1024, out_features=512, bias=True)
    (5): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False

In [5]:
all_preds = []

with torch.no_grad():
    for X_batch in test_dl:
        X_batch = X_batch.to(device)
        preds = model0(X_batch)
        all_preds.append(preds.cpu())

y_pred_test1 = np.expm1(torch.cat(all_preds).numpy())

all_preds = []

with torch.no_grad():
    for X_batch in test_dl:
        X_batch = X_batch.to(device)
        preds = model1(X_batch)
        all_preds.append(preds.cpu())

y_pred_test2 = np.expm1(torch.cat(all_preds).numpy())

all_preds = []

with torch.no_grad():
    for X_batch in test_dl:
        X_batch = X_batch.to(device)
        preds = model2(X_batch)
        all_preds.append(preds.cpu())

y_pred_test3 = np.expm1(torch.cat(all_preds).numpy())

all_preds = []

with torch.no_grad():
    for X_batch in test_dl:
        X_batch = X_batch.to(device)
        preds = model3(X_batch)
        all_preds.append(preds.cpu())

y_pred_test4 = np.expm1(torch.cat(all_preds).numpy())

all_preds = []

with torch.no_grad():
    for X_batch in test_dl:
        X_batch = X_batch.to(device)
        preds = model4(X_batch)
        all_preds.append(preds.cpu())

y_pred_test5 = np.expm1(torch.cat(all_preds).numpy())

all_preds = []
with torch.no_grad():
    for X_batch in test_dl:
        X_batch = X_batch.to(device)
        preds = model5(X_batch)
        all_preds.append(preds.cpu())

y_pred_test6 = np.expm1(torch.cat(all_preds).numpy())

y_pred_test = (y_pred_test1 + y_pred_test2 + y_pred_test3 + y_pred_test4 + y_pred_test5 + y_pred_test6) / 6

In [ ]:
import pandas as pd

submission_df = pd.DataFrame({
    "sample_id": test_df["sample_id"],
    "price": y_pred_test
})

submission_df.to_csv("test_out.csv", index=False)
print("💾 Predictions saved to test_out.csv")

💾 Predictions saved to submission 0.3 weighted.csv
